In [10]:
import utils
import const
import models

import re
import pandas as pd
import torch
from itertools import combinations
from transformers import AutoTokenizer

In [2]:
test_doc_id = 'CNN_20130322_314'

In [3]:
doc_df = utils.get_docs([test_doc_id])

In [4]:
doc_df.iloc[0]['text']

"President Barack Obama [E1]arrived[/E1] in refugee-flooded Jordan on [TIMEX3]Friday[/TIMEX3] after [E2]scoring[/E2] a diplomatic [E3]coup[/E3] just before [E4]leaving[/E4] Israel when Prime Minister Benjamin Netanyahu [E5]apologized[/E5] to Turkey for a [TIMEX3]2010[/TIMEX3] commando [E6]raid[/E6] that [E7]killed[/E7] nine activists on a Turkish vessel in a Gaza-bound flotilla.\n\nThe apology, long [E8]sought[/E8] by Turkish Prime Minister Recep Erdogan, [E9]eased[/E9] strained feelings between Turkey and Israel, two vital U.S. allies in the Middle East.\n\nIt [E10]happened[/E10] in a phone [E29]call[/E29] to Erdogan during a final [E11]meeting[/E11] between Obama and Netanyahu at Ben Gurion International Airport in Tel Aviv [TIMEX3]minutes[/TIMEX3] before Air Force One [E12]departed[/E12] for Jordan to [E13]complete[/E13] the president's Middle East [E30]swing[/E30]. It is the first foreign [E14]trip[/E14] of his [TIMEX3]second term[/TIMEX3].\n\nA statement from Netanyahu's office [E

In [5]:
rows = []
pattern = re.compile(r"\[E(\d+)\](.*?)\[/E\1\]")

for docid, row in doc_df.iterrows():
    text = row.get("text", "")
    events = [(f"E{eiid}", verb.strip()) for eiid, verb in pattern.findall(text)]
    
    for (eiid1, verb1), (eiid2, verb2) in combinations(events, 2):
        rows.append(
            {
                "docid": docid,
                "verb1": verb1,
                "verb2": verb2,
                "eiid1": eiid1,
                "eiid2": eiid2,
            }
        )

event_pairs_df = pd.DataFrame(rows, columns=["docid", "verb1", "verb2", "eiid1", "eiid2"])
event_pairs_df.shape

(528, 5)

In [6]:
relations_df = utils.create_context_windows(event_pairs_df, doc_df)
relations_df.head()

,docid,verb1,verb2,eiid1,eiid2,context_window
0,CNN_20130322_314,arrived,scoring,E1,E2,President Barack Obama [T1]arrived[/T1] in ref...
1,CNN_20130322_314,arrived,coup,E1,E3,President Barack Obama [T1]arrived[/T1] in ref...
2,CNN_20130322_314,arrived,leaving,E1,E4,President Barack Obama [T1]arrived[/T1] in ref...
3,CNN_20130322_314,arrived,apologized,E1,E5,President Barack Obama [T1]arrived[/T1] in ref...
4,CNN_20130322_314,arrived,raid,E1,E6,President Barack Obama [T1]arrived[/T1] in ref...


In [7]:
state_dict = torch.load('./temp_rel_roberta.pt')
tokenizer = models.create_temp_rel_tokenizer()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [8]:
loader = utils.create_evaluation_loader(relations_df, tokenizer, batch_size=16)

input_ids shape: torch.Size([528, 276])
attention_mask shape: torch.Size([528, 276])


In [11]:
from transformers import AutoModelForSequenceClassification

all_preds, all_true = [], []

num_labels = len(const.relation2id)
model = models.TemporalRelationsModel(num_labels=num_labels, tokenizer=tokenizer)
model.load_state_dict(state_dict, strict=False)
model = model.to(device)
model.eval()

with torch.no_grad():
    for input_ids, attention_mask in loader:
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)

        # Keep only samples that contain both target markers
        has_t1 = (input_ids == model.t1_id).any(dim=1)
        has_t2 = (input_ids == model.t2_id).any(dim=1)
        valid = has_t1 & has_t2
        if not valid.any():
            continue

        input_ids = input_ids[valid]
        attention_mask = attention_mask[valid]

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits if hasattr(outputs, "logits") else outputs
        preds = torch.argmax(logits, dim=-1).cpu()

        all_preds.append(preds)

y_pred = torch.cat(all_preds).numpy() if all_preds else torch.empty(0, dtype=torch.long).numpy()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: FacebookAI/roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To dis

In [13]:
relations_df["predicted_relation"] = [const.id2relation[pred] for pred in y_pred]

In [17]:
relations_df.tail(10)

,docid,verb1,verb2,eiid1,eiid2,context_window,predicted_relation
518,CNN_20130322_314,said,agreed,E24,E25,There also was no word on whether the once-clo...,BEFORE
519,CNN_20130322_314,said,cancellation,E24,E26,There also was no word on whether the once-clo...,BEFORE
520,CNN_20130322_314,said,omitted,E24,E27,There also was no word on whether the once-clo...,VAGUE
521,CNN_20130322_314,said,action,E24,E28,There also was no word on whether the once-clo...,VAGUE
522,CNN_20130322_314,agreed,cancellation,E25,E26,There also was no word on whether the once-clo...,BEFORE
523,CNN_20130322_314,agreed,omitted,E25,E27,There also was no word on whether the once-clo...,VAGUE
524,CNN_20130322_314,agreed,action,E25,E28,There also was no word on whether the once-clo...,AFTER
525,CNN_20130322_314,cancellation,omitted,E26,E27,There also was no word on whether the once-clo...,VAGUE
526,CNN_20130322_314,cancellation,action,E26,E28,There also was no word on whether the once-clo...,AFTER
527,CNN_20130322_314,omitted,action,E27,E28,Turkey had been prosecuting four Israeli soldi...,VAGUE


In [15]:
doc_df.iloc[0]['text']

"President Barack Obama [E1]arrived[/E1] in refugee-flooded Jordan on [TIMEX3]Friday[/TIMEX3] after [E2]scoring[/E2] a diplomatic [E3]coup[/E3] just before [E4]leaving[/E4] Israel when Prime Minister Benjamin Netanyahu [E5]apologized[/E5] to Turkey for a [TIMEX3]2010[/TIMEX3] commando [E6]raid[/E6] that [E7]killed[/E7] nine activists on a Turkish vessel in a Gaza-bound flotilla.\n\nThe apology, long [E8]sought[/E8] by Turkish Prime Minister Recep Erdogan, [E9]eased[/E9] strained feelings between Turkey and Israel, two vital U.S. allies in the Middle East.\n\nIt [E10]happened[/E10] in a phone [E29]call[/E29] to Erdogan during a final [E11]meeting[/E11] between Obama and Netanyahu at Ben Gurion International Airport in Tel Aviv [TIMEX3]minutes[/TIMEX3] before Air Force One [E12]departed[/E12] for Jordan to [E13]complete[/E13] the president's Middle East [E30]swing[/E30]. It is the first foreign [E14]trip[/E14] of his [TIMEX3]second term[/TIMEX3].\n\nA statement from Netanyahu's office [E